# 01 — Fits (3 arms × 3 seeds × 28 nested fits)

`exp2_aug` recipe, provider targets, `TrainConfig()`. Order: `exp4_full` (3 seeds), `exp4_touch`, `exp4_aux`. Each fit saves its weights, its final training NLL, and predicts both views of its held-out matches. Cached by fingerprint; a per-job lock lets an optional second process (`fit_worker.py`) share the work without fitting any job twice. Progress in `results/fit_log.txt`.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp4_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_4')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp4_common as C
import exp1_common as C1
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp4_protocol as P, exp1_net as N
print('device', N.DEVICE, '| config', N.TrainConfig().as_dict())

PREREG verified 254e2e216cb05154 | frozen 2026-11-09T17:09:15+00:00
device cuda | config {'window': 300, 'batch': 32, 'epochs': 14, 'coverage': 2.0, 'lr': 0.002, 'weight_decay': 0.0001, 'width': 64, 'hidden': 64, 'boundary_weight': 0.5, 'boundary_pos_weight': 3.0, 'mirror': True, 'seed': 20261108}


## Reproduction check (1 fit, report only)
exp_4's training loop with aux off and no touch columns, exp2_aug's recipe and seed, outer fold of the first match, against exp_2's cached `exp2_aug` prediction. Bit-exact expected; max |Δp| > 1e-3 stops the run.

In [2]:
log_path = C.RESULTS / 'fit_log.txt'
def log(message, **_):
    print(message, flush=True)
    with log_path.open('a') as handle:
        handle.write(message + '\n')
path = C.RESULTS / 'repro_check.json'
if path.exists():
    repro = C1.read_json(path)
else:
    t0 = time.time()
    repro = P.reproduction_check()
    repro['seconds'] = round(time.time() - t0, 1)
    C1.write_json(path, repro)
    log(f"REPRO CHECK {repro['held_out']} bit_exact={repro['bit_exact']} {repro['seconds']}s")
print(json.dumps(repro, indent=1))
assert max(repro[v]['max_abs_probability_diff'] for v in C.VIEWS) <= 1e-3

REPRO CHECK J03WMX bit_exact=True 59.9s


{
 "held_out": "J03WMX",
 "fits": 1,
 "original": {
  "max_abs_probability_diff": 0.0,
  "max_abs_boundary_diff": 0.0,
  "bit_exact": true
 },
 "target_like": {
  "max_abs_probability_diff": 0.0,
  "max_abs_boundary_diff": 0.0,
  "bit_exact": true
 },
 "bit_exact": true,
 "seconds": 59.9
}


In [3]:
t0 = time.time()
for arm in P.FIT_ORDER:
    remaining = sum(1 for k, *_ in P.P1.job_keys() if not (P.prediction_folder(arm) / f'{k}.npz').exists())
    assert P.fit_count() + 1 + remaining <= C.FIT_CAP, 'fit cap reached'
    start = time.time()
    P.ensure_predictions(arm, log=log)
    log(f'ARM DONE {arm} {time.time() - start:.0f}s, fits cached {P.fit_count()}')
log(f'ALL FITS DONE {time.time() - t0:.0f}s, exp_4 proxy fits cached {P.fit_count()} (+1 reproduction check)')

FIT exp4_full pair_J03WMX_J03WN1 55.7s train_nll 0.1171


FIT exp4_full pair_J03WMX_J03WOH 55.5s train_nll 0.1277


FIT exp4_full pair_J03WMX_J03WOY 55.6s train_nll 0.1232


FIT exp4_full pair_J03WMX_J03WPY 55.0s train_nll 0.1267


FIT exp4_full pair_J03WMX_J03WQQ 55.6s train_nll 0.1231


FIT exp4_full pair_J03WMX_J03WR9 54.9s train_nll 0.1234


... (252 outputs left out)

FIT exp4_aux_s2 outer_J03WQQ 65.4s train_nll 0.1152


FIT exp4_aux_s2 outer_J03WR9 64.7s train_nll 0.1175


ARM DONE exp4_aux_s2 1616s, fits cached 252


ALL FITS DONE 14492s, exp_4 proxy fits cached 252 (+1 reproduction check)


In [4]:
train = pd.DataFrame([r for arm in P.ARMS for r in P.train_nll_rows(arm)])
print(train.groupby('arm').train_nll.agg(['mean', 'min', 'max', 'count']).round(4).to_string())

                 mean     min     max  count
arm                                         
exp4_aux       0.1180  0.1116  0.1269     28
exp4_aux_s1    0.1174  0.1102  0.1252     28
exp4_aux_s2    0.1190  0.1112  0.1264     28
exp4_full      0.1245  0.1152  0.1328     28
exp4_full_s1   0.1267  0.1181  0.1344     28
exp4_full_s2   0.1226  0.1147  0.1299     28
exp4_touch     0.1249  0.1162  0.1339     28
exp4_touch_s1  0.1241  0.1165  0.1296     28
exp4_touch_s2  0.1221  0.1146  0.1289     28
